# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 protein abundance dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined using a Croissant schema and is accessible via a public URL.

In [ ]:
# Install mlcroissant if necessary
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings

# Silence common pandas and mlcroissant warnings for cleanliness
warnings.filterwarnings("ignore")

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

In [ ]:
# List all record sets with their @id and description
record_sets = list(dataset.record_sets)
print("Available record sets (by @id):")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('description', 'No description')}")

# Preview fields for each record set
print("\nFields for each record set:")
for rs in record_sets:
    fields = rs.get('field', [])
    print(f"\nRecordSet @id: {rs['@id']}")
    for field in fields:
        if isinstance(field, dict):
            print(f"  - {field.get('@id')} (name: {field.get('name')}, type: {field.get('dataType')})")
        else:
            print(f"  - {field}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All Croissant entities are referenced by their `@id`.

In [ ]:
# Extract all available record sets and load into DataFrames
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}
for rec_id in record_set_ids:
    print(f"Reading records for: {rec_id}")
    records = list(dataset.records(record_set=rec_id))
    if len(records) > 0:
        dataframes[rec_id] = pd.DataFrame(records)
    else:
        print(f"No data records found for record set {rec_id}")
print("\nExtracted DataFrames:")
for rec_id in dataframes:
    print(f"- {rec_id}: columns={dataframes[rec_id].columns.tolist()}")

# For demonstration, show the first DataFrame's columns and head
if len(dataframes) > 0:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nPreview of main DataFrame ({main_record_set_id}):")
    print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Explore, filter, transform, and group the data using key fields.
- For demonstration, we select numeric and grouping fields from the first record set discovered in the previous step.

In [ ]:
# Automated field selection: try to pick a numeric and a group field
import numpy as np

df = dataframes[main_record_set_id]
print(f"Columns in {main_record_set_id}:", df.columns.tolist())

# Try to choose a numeric field for demonstration (e.g., 'cr:field:MW')
numeric_field = None
group_field = None
for col in df.columns:
    if np.issubdtype(df[col].dtype, np.number) and numeric_field is None:
        numeric_field = col
    elif df[col].dtype == "object" and group_field is None:
        group_field = col
if numeric_field is None:
    # fallback: search columns containing 'MW' (common for molecular weight)
    for col in df.columns:
        if 'MW' in col or 'mw' in col:
            numeric_field = col
            break
if group_field is None:
    for col in df.columns:
        if 'accession' in col.lower() or 'id' in col.lower():
            group_field = col
            break
print(f"\nSelected numeric field: {numeric_field}")
print(f"Selected group field: {group_field}")

# Convert numeric_field to float if not already
if not np.issubdtype(df[numeric_field].dtype, np.number):
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

threshold = df[numeric_field].quantile(0.75)
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records where {numeric_field} > {threshold:.2f} (top quartile): {filtered_df.shape[0]} rows")
display(filtered_df[[numeric_field, group_field]].head())

# Normalize the numeric field in the filtered data
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()

print("\nNormalized sample:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group data by group_field and get mean statistics
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped data by {group_field}, mean({numeric_field}):")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and how it relates to the grouping field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df[numeric_field].dropna(), kde=True, ax=ax[0])
ax[0].set_title(f"Distribution of {numeric_field}")
ax[0].set_xlabel(numeric_field)

# If enough unique groups, show average value per group
if group_field and group_field in df.columns and df[group_field].nunique() < 50:
    group_means = df.groupby(group_field)[numeric_field].mean().sort_values()
    group_means.plot(kind="bar", ax=ax[1])
    ax[1].set_title(f"Mean {numeric_field} by {group_field}")
    ax[1].set_ylabel(f"Mean {numeric_field}")
else:
    # Alternate: scatter plot (top 100 by group)
    short_df = df[[numeric_field, group_field]].dropna()
    if short_df.shape[0] > 100:
        short_df = short_df.sample(100, random_state=0)
    sns.scatterplot(x=group_field, y=numeric_field, data=short_df, ax=ax[1])
    ax[1].set_title(f"{numeric_field} vs. {group_field} (sample)")

plt.tight_layout()
plt.show()

## 6. Conclusion
- Using the `mlcroissant` library, we loaded and explored the extracellular vesicle mass spectrometry dataset using only Croissant `@id` for all references.
- We extracted protein abundance and related fields, filtered for high-value records, normalized data, and visualized distributions.
- This approach can be extended to any FAIR^2 or Croissant-compliant dataset for transparent, reproducible research workflows.

For more details, see documentation at https://mlcommons.org/croissant and https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json.